In [ ]:
import boto3
import sagemaker

role = sagemaker.get_execution_role()
print('Role:', role)

s3 = boto3.client('s3')
for bucket in s3.list_buckets()['Buckets']:
    print('Bucket:', bucket['Name'])

In [ ]:
import boto3

BUCKET = 'sagemaker-us-east-1-488236761809'
MODEL_S3_KEY = 'models/final_model_pipeline.pkl'

s3 = boto3.client('s3')
s3.download_file(BUCKET, MODEL_S3_KEY, 'final_model_pipeline.pkl')
print('Model downloaded.')

In [ ]:
import tarfile

with tarfile.open('model.tar.gz', 'w:gz') as tar:
    tar.add('final_model_pipeline.pkl', arcname='final_model_pipeline.pkl')

print('model.tar.gz created.')

In [ ]:
s3 = boto3.client('s3')
s3.upload_file('model.tar.gz', BUCKET, 'model/model.tar.gz')
print('Uploaded model.tar.gz to S3.')

In [ ]:
from inference import model_fn
import os
os.makedirs('model', exist_ok=True)
import shutil
shutil.copy('final_model_pipeline.pkl', 'model/final_model_pipeline.pkl')
model = model_fn('model')
print('Model loaded successfully:', type(model))

In [ ]:
import boto3
import sagemaker
from sagemaker.sklearn.model import SKLearnModel

BUCKET         = 'sagemaker-us-east-1-488236761809'
MODEL_S3_KEY   = 'model/model.tar.gz'
ENDPOINT_NAME  = 'UAS-endpoint'
REGION         = 'us-east-1'
INSTANCE_TYPE  = 'ml.m5.large'
FRAMEWORK_VERSION = '1.2-1'

def get_lab_role_arn():
    iam = boto3.client('iam')
    return iam.get_role(RoleName='LabRole')['Role']['Arn']

def main():
    boto3.setup_default_session(region_name=REGION)
    sm_session = sagemaker.Session()
    role_arn   = get_lab_role_arn()
    model_s3_uri = f's3://{BUCKET}/{MODEL_S3_KEY}'

    print('Role:     ', role_arn)
    print('Model URI:', model_s3_uri)
    print('Endpoint: ', ENDPOINT_NAME)

    model = SKLearnModel(
        model_data=model_s3_uri,
        role=role_arn,
        entry_point='inference.py',
        framework_version=FRAMEWORK_VERSION,
        sagemaker_session=sm_session,
    )

    print('\nDeploying endpoint (5-8 minutes)...')
    model.deploy(
        initial_instance_count=1,
        instance_type=INSTANCE_TYPE,
        endpoint_name=ENDPOINT_NAME
    )

    print('Testing endpoint...')
    runtime = boto3.client('sagemaker-runtime', region_name=REGION)
    import json
    sample = {"instances": [[33.0, 36938.82, 3080.35, 5, 5, 13, 3, 18, 14, 10.25, 5, 998.81, 32.32, 62.77, 137.15, 344.39, 227, 3, 'January', 'Lawyer', 'Standard', 'Yes', 'Low_spent_Small_value_payments']]}
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType='application/json',
        Accept='application/json',
        Body=json.dumps(sample),
    )
    print('Smoke test response:')
    print(response['Body'].read().decode('utf-8'))

main()

In [ ]:
import boto3

sm_client = boto3.client('sagemaker', region_name='us-east-1')
ENDPOINT_NAME = 'UAS-endpoint'

try:
    sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print('Endpoint deleted.')
except Exception as e:
    print(f'No endpoint to delete: {e}')

try:
    sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print('Endpoint config deleted.')
except Exception as e:
    print(f'No config to delete: {e}')